## 1. 라이브러리 불러오기

In [1]:
from flask import Flask, flash, session, send_from_directory, send_file, Response, redirect, url_for, request, session, abort, jsonify, render_template
from werkzeug.utils import secure_filename
from pymongo import MongoClient
from bson import objectid
import pandas as pd
import os
import requests

## 2. CSV 포맷의 데이터를 MongoDB에 저장하기

### 2-1. CSV 포맷으로 저장된 데이터를 데이터베이스에 저장하기

In [8]:
# MongoDB 클라이언트 설정
client = MongoClient('mongodb://localhost:27017/')  # MongoDB 서버 주소
db = client['db_for_csv']  # 데이터베이스 이름
collection = db['data_collection1']  # 컬렉션 이름

# CSV 파일 읽기
csv_file_path = 'csv-files-upload\RT.34224607-100W-36H-20190620224719.csv'
data = pd.read_csv(csv_file_path)

# DataFrame을 딕셔너리 형태로 변환
data_dict = data.to_dict("records")

# MongoDB에 데이터 삽입
collection.insert_many(data_dict)

print(f"Data from {csv_file_path} has been inserted into MongoDB.")

Data from csv-files-upload\RT.34224607-100W-36H-20190620224719.csv has been inserted into MongoDB.


### 2-2. 데이터베이스로부터 CSV 파일 포맷으로 데이터 불러오기

In [9]:
# MongoDB 클라이언트 설정
client = MongoClient('mongodb://localhost:27017/')  # MongoDB 서버 주소
db = client['db_for_csv']  # 데이터베이스 이름
collection = db['data_collection1']  # 컬렉션 이름

# MongoDB에서 데이터 조회
data = list(collection.find())

# '_id' 필드를 제거 (선택 사항)
for record in data:
    if '_id' in record:
        del record['_id']

# DataFrame으로 변환
data_df = pd.DataFrame(data)

# CSV 파일로 저장
csv_file = csv_file_path.split('\\')[-1]
csv_output_path = 'downloaded_'+ csv_file
data_df.to_csv(csv_output_path, index=False)

print(f"Data from MongoDB has been written to {csv_output_path}.")

Data from MongoDB has been written to downloaded_RT.34224607-100W-36H-20190620224719.csv.


## 3. HDF 포맷의 데이터를 MongoDB에 저장하기

### 3-1. HDF 포맷으로 저장된 데이터를 데이터베이스에 저장하기
- MongoDB의 기본 BSON 문서 크기 제한은 16MB이다. 이 제한을 초과하는 데이터를 처리하려면 여러 문서로 분할하여 저장해야 합니다. 하지만 분할하여 저장하는 방법을 일반화하기는 어렵다. 대용량의 데이터를 관리하기 위해서는 다른 대안이 필요하다.

In [17]:
import h5py
import numpy as np
from pymongo import MongoClient

# MongoDB 클라이언트 설정
client = MongoClient('mongodb://localhost:27017/')  # MongoDB 서버 주소
db = client['db_for_hdf']  # 데이터베이스 이름
collection = db['data_collection1']  # 컬렉션 이름

# HDF5 파일 읽기
hdf5_file_path = 'hdf-files-upload\Product-34224607-100W-6D-20190620224719.hdf5'

def read_hdf5_to_dict(file_path):
    data_dict = {}
    with h5py.File(file_path, 'r') as f:
        for key in f.keys():
            data = f[key][()]
            if isinstance(data, np.ndarray):
                data_dict[key] = data.tolist()  # numpy 배열을 리스트로 변환
            else:
                data_dict[key] = data
    return data_dict

# HDF5 파일의 데이터를 딕셔너리로 변환
data_dict = read_hdf5_to_dict(hdf5_file_path)

# MongoDB에 데이터 삽입
collection.insert_one(data_dict)

print(f"Data from {hdf5_file_path} has been inserted into MongoDB.")

DocumentTooLarge: BSON document too large (47933078 bytes) - the connected server supports BSON document sizes up to 16793598 bytes.

### 3-2. 파일 이름만 데이터베이스에 저장하여 관리하기
- 아래의 코드는 파일 이름을 분리하여 데이터베이스에 저장하는 방법이다.

In [21]:
DATA_PATH = os.getcwd() + os.path.sep + 'data'
URL = 'http://192.168.45.224:27017'

mongoclient = MongoClient('192.168.45.224', 27017)

if not 'db_for_hdf' in mongoclient.list_database_names():
    mongoclient['db_for_hdf'].create_collection('data_collection1')
    
cores = mongoclient['db_for_hdf']['data_collection1']

# HDF5 파일 읽기
hdf5_file_path = 'hdf-files-upload\Product-34224607-100W-6D-20190620224719.hdf5'
hdf5_file = hdf5_file_path.split('\\')[-1]

def upload(filename):
    tmp, sn, type, dim, dt = filename.split('.')[0].split('-')
    item = {
        'sn': sn,
        'type': type,
        'dim': dim,
        'date': dt,
        'data': filename
    }
    cores.insert_one(item)

upload(hdf5_file)

- 아래의 코드는 id를 바탕으로 데이터베이스의 데이터를 삭제하는 방법이다.

In [19]:
def delete(id):
    cores.delete_one({'_id': objectid.ObjectId(id)})
    
delete('664627965a53be3dc09ac44a')

- 아래의 코드는 type을 바탕으로 ObjectID를 찾아내는 방법이다.

In [20]:
cur = cores.find({'type': '100W'})
for c in cur:
    print(c['_id'])

67ffadf3ac04a34733905473
67ffae1eac04a34733905475
67ffae41ac04a34733905477
67ffaecbac04a34733905479
67ffaedfac04a3473390547d


## 4. Flask 서버와 MongoDB를 이용하여 HDF5 데이터 관리하기

### 4-1. HDF 파일을 Flask 서버에 저장하기
- 아래의 코드는 HDF 파일을 Flask 서버의 "/upload" 엔드포인트로 업로드한다.

In [26]:
url = 'http://127.0.0.1:5000/upload'
file_path = 'hdf-files-upload\Product-34224607-100W-6D-20190620224719.hdf5'

with open(file_path, 'rb') as f:
    files = {'file': f}
    response = requests.post(url, files=files)

print(response.json())

{'message': 'File Product-34224607-100W-6D-20190620224719.hdf5 has been uploaded and saved.'}


- 아래의 코드는 Flask 서버의 "/files" 엔드포인트로 GET 요청을 보내 파일 목록을 조회한다.

In [27]:
url = 'http://127.0.0.1:5000/files'
response = requests.get(url)

print(response.json())

['Product-34224607-100W-6D-20190620224719.hdf5']


- 아래의 코드는 Flask 서버의 "/download/<filename>" 엔드포인트로 GET 요청을 보내 파일을 다운로드한다.

In [28]:
filename = 'Product-34224607-100W-6D-20190620224719.hdf5'
url = f'http://127.0.0.1:5000/download/{filename}'
response = requests.get(url)

with open(f'downloaded_{filename}', 'wb') as f:
    f.write(response.content)

print(f'File {filename} downloaded.')

File Product-34224607-100W-6D-20190620224719.hdf5 downloaded.
